In [6]:
"""Demo: visualize a simple ballistic Apollo-like entry to verify conventions."""

import numpy as np
from reentrykit.trajectory import Vehicle, InitialState, simulate
from reentrykit.planet import EARTH, EARTH_NON_ROTATING
from reentrykit.visualization import plot_trajectory_3d, plot_trajectory_summary


# Simple test: ballistic Apollo-like capsule, entering at 25°N, heading west
# V-B-C convention: psi = pi means due west
apollo_test = Vehicle.from_mass_area_cd(
    mass=5357.0, reference_area=12.0, drag_coefficient=1.2,
    lift_to_drag_ratio=0.0,    # ballistic for simplicity
    nose_radius=4.661,
)

apollo_state = InitialState(
    altitude=121_920.0,
    velocity=11_137.0,
    flight_path_angle=np.deg2rad(-6.93),
    heading=np.pi,              # V-B-C: due west
    latitude=np.deg2rad(25.0),
    longitude=np.deg2rad(0.0),
)

result = simulate(
    apollo_test, apollo_state,
    planet=EARTH, max_time=1500.0, dt_output=1.0,
)

print(f"Flight time: {result.time[-1]:.0f} s")
print(f"Final lat:   {np.rad2deg(result.latitude[-1]):.2f}°")
print(f"Final lon:   {np.rad2deg(result.longitude[-1]):.2f}°")
print(f"If heading was truly west, final longitude should be less than 0° (westward drift)")

# Show the 3D view
fig = plot_trajectory_3d(apollo_test_result := result, apollo_state, planet_name="Earth")
fig.show()

Flight time: 304 s
Final lat:   24.69°
Final lon:   -10.12°
If heading was truly west, final longitude should be less than 0° (westward drift)


In [7]:
"""Compare Apollo high-res under different rotating-Earth configurations."""

# Use the high-res L/D schedule (same one Apollo notebook uses)
# Rebuild it here standalone for clarity
import numpy as np
from reentrykit.trajectory import Vehicle, InitialState, simulate
from reentrykit.planet import EARTH, EARTH_NON_ROTATING
from reentrykit.visualization import plot_trajectory_summary

# Apollo vehicle with constant Cd, constant L/D magnitude 0.368 (no bank schedule)
# — simplified for visualization, not for peak-g accuracy
apollo_viz = Vehicle.from_mass_area_cd(
    mass=5357.0, reference_area=12.0, drag_coefficient=1.2,
    lift_to_drag_ratio=0.368,       # lift-up constant (no bank)
    nose_radius=4.661,
)

# Compare two headings
for heading_label, heading_rad in [("west (ψ=π)", np.pi), ("east (ψ=0)", 0.0)]:
    for planet_label, planet in [("rotating", EARTH), ("non-rotating", EARTH_NON_ROTATING)]:
        state = InitialState(
            altitude=121_920.0,
            velocity=11_137.0,
            flight_path_angle=np.deg2rad(-6.93),
            heading=heading_rad,
            latitude=np.deg2rad(25.0),
            longitude=0.0,
        )
        r = simulate(apollo_viz, state, planet=planet,
                     max_time=1500.0, dt_output=1.0)
        dV_dt = np.gradient(r.velocity, r.time)
        peak_g = -dV_dt.min() / 9.80665

        print(
            f"Heading {heading_label:<16} | {planet_label:<13} | "
            f"peak g = {peak_g:.2f} | "
            f"final lon = {np.rad2deg(r.longitude[-1]):.1f}° | "
            f"termination = {r.termination_reason}"
        )

Heading west (ψ=π)       | rotating      | peak g = 8.98 | final lon = -65.0° | termination = Ground impact
Heading west (ψ=π)       | non-rotating  | peak g = 7.35 | final lon = -57.0° | termination = Skip-out above extended atmosphere ceiling
Heading east (ψ=0)       | rotating      | peak g = 5.71 | final lon = 40.9° | termination = Skip-out above extended atmosphere ceiling
Heading east (ψ=0)       | non-rotating  | peak g = 7.35 | final lon = 57.0° | termination = Skip-out above extended atmosphere ceiling


In [8]:
state_east_rot = InitialState(
    altitude=121_920.0,
    velocity=11_137.0,
    flight_path_angle=np.deg2rad(-6.93),
    heading=0.0,                    # east
    latitude=np.deg2rad(25.0),
    longitude=0.0,
)
r_east = simulate(apollo_viz, state_east_rot, planet=EARTH,
                  max_time=1500.0, dt_output=1.0)
fig = plot_trajectory_summary(r_east, state_east_rot,
                              planet_name="Earth (rotating)",
                              title="Apollo viz: east heading, rotating Earth")
fig.show()

In [9]:
print("Heading sweep (rotating Earth):")
print(f"{'Heading (V-B-C)':<20} {'Aerospace (from N CW)':<25} {'Peak g':<10}")
print("-" * 60)
for heading_vbc_deg in [0, 45, 90, 135, 180, 225, 270, 315]:
    heading_vbc = np.deg2rad(heading_vbc_deg)
    # Convert to aerospace convention for display
    aerospace_from_N = (90.0 - heading_vbc_deg) % 360
    state = InitialState(
        altitude=121_920.0,
        velocity=11_137.0,
        flight_path_angle=np.deg2rad(-6.93),
        heading=heading_vbc,
        latitude=np.deg2rad(25.0),
        longitude=0.0,
    )
    r = simulate(apollo_viz, state, planet=EARTH,
                 max_time=1500.0, dt_output=1.0)
    dV_dt = np.gradient(r.velocity, r.time)
    peak_g = -dV_dt.min() / 9.80665
    print(f"ψ = {heading_vbc_deg:>3}°{'':<14} {aerospace_from_N:>3.0f}° "
          f"{'':<15} {peak_g:.2f}")

Heading sweep (rotating Earth):
Heading (V-B-C)      Aerospace (from N CW)     Peak g    
------------------------------------------------------------
ψ =   0°                90°                 5.71
ψ =  45°                45°                 6.18
ψ =  90°                 0°                 7.32
ψ = 135°               315°                 8.49
ψ = 180°               270°                 8.98
ψ = 225°               225°                 8.49
ψ = 270°               180°                 7.32
ψ = 315°               135°                 6.17
